# Pandas: Selecting and Filtering


**What you'll learn:**
1. `.loc` for label-based selection
2. `.iloc` for position-based selection
3. Boolean masking on DataFrames
4. The `.query()` method for SQL-like filtering
5. `isin()`, `between()`, and a few other handy filters
6. `SettingWithCopyWarning` and how to avoid it

A few terms before we begin:
- **Selecting**: pulling out specific rows or columns from a DataFrame.
- **Filtering**: keeping only the rows that satisfy a condition.
- **Indexer**: a pandas object (`.loc`, `.iloc`) used to access data by labels or positions.
- **Label**: the name attached to a row (in the index) or a column.
- **Position**: the integer location of a row or column, starting from 0.

---

In [2]:
import pandas as pd
import numpy as np

# Set up a sample dataset we'll use throughout
data = {
    'name':   ['Ali', 'Sara', 'Ahmed', 'Fatima', 'Hassan', 'Ayesha', 'Bilal', 'Maryam'],
    'age':    [22, 25, 19, 28, 21, 23, 20, 24],
    'city':   ['Lahore', 'Karachi', 'Islamabad', 'Lahore', 'Karachi', 'Peshawar', 'Lahore', 'Karachi'],
    'gpa':    [3.5, 3.8, 2.9, 3.6, 3.1, 3.7, 2.8, 3.9],
    'major':  ['CS', 'EE', 'CS', 'ME', 'CS', 'EE', 'ME', 'CS']
}
df = pd.DataFrame(data)
print(df)

     name  age       city  gpa major
0     Ali   22     Lahore  3.5    CS
1    Sara   25    Karachi  3.8    EE
2   Ahmed   19  Islamabad  2.9    CS
3  Fatima   28     Lahore  3.6    ME
4  Hassan   21    Karachi  3.1    CS
5  Ayesha   23   Peshawar  3.7    EE
6   Bilal   20     Lahore  2.8    ME
7  Maryam   24    Karachi  3.9    CS


## `.loc` and `.iloc` - The Two Indexers

These are the two most important selection tools in pandas. They look alike but mean different things.

### definitions

> "`.loc` - Access a group of rows and columns by label(s) or a boolean array." - pandas docs

> "`.iloc` - Purely integer-location based indexing for selection by position." - pandas docs

| Indexer | What it uses | Example |
|---|---|---|
| `.loc[]`  | **Labels** (the index and column names) | `df.loc[2, 'name']` |
| `.iloc[]` | **Positions** (integer locations) | `df.iloc[2, 0]` |

### `.loc` - by label

In [3]:
# Get one row by its INDEX LABEL (here, label = 2)
print("Row with index label 2:")
print(df.loc[2])
print(f"\nType: {type(df.loc[2]).__name__}")

Row with index label 2:
name         Ahmed
age             19
city     Islamabad
gpa            2.9
major           CS
Name: 2, dtype: object

Type: Series


In [4]:
# Get a single cell
print("Row 2, column 'name':", df.loc[2, 'name'])

# Get multiple rows + columns
print("\nRows 1-3, columns name + gpa:")
print(df.loc[1:3, ['name', 'gpa']])

# Note: with .loc, the end of a range IS INCLUDED. Different from Python lists!

Row 2, column 'name': Ahmed

Rows 1-3, columns name + gpa:
     name  gpa
1    Sara  3.8
2   Ahmed  2.9
3  Fatima  3.6


In [5]:
# Just specific rows, all columns
print("Rows 0, 2, 5:")
print(df.loc[[0, 2, 5]])

# All rows, specific columns
print("\nAll names and ages:")
print(df.loc[:, ['name', 'age']])

Rows 0, 2, 5:
     name  age       city  gpa major
0     Ali   22     Lahore  3.5    CS
2   Ahmed   19  Islamabad  2.9    CS
5  Ayesha   23   Peshawar  3.7    EE

All names and ages:
     name  age
0     Ali   22
1    Sara   25
2   Ahmed   19
3  Fatima   28
4  Hassan   21
5  Ayesha   23
6   Bilal   20
7  Maryam   24


### Key difference: `.loc` includes the endpoint

```python
df.loc[1:3]    # rows with index labels 1, 2, AND 3
df.iloc[1:3]   # rows at positions 1 and 2 (3 is NOT included)
```

This catches almost everyone the first time. The reason: `.loc` works on labels, and labels aren't always integers (they could be strings, dates, etc.), so label-based slicing includes both endpoints. `.iloc` follows the usual Python slicing rule and excludes the endpoint.

### `.iloc` - by integer position

In [6]:
# Get one row by POSITION
print("Row at position 2:")
print(df.iloc[2])

# Single cell
print("\nRow 2, column 0:", df.iloc[2, 0])

# Multiple rows + columns
print("\nRows 1-3 (4 excluded), first 2 columns:")
print(df.iloc[1:4, 0:2])

# Specific positions
print("\nRows 0, 2, 5:")
print(df.iloc[[0, 2, 5]])

Row at position 2:
name         Ahmed
age             19
city     Islamabad
gpa            2.9
major           CS
Name: 2, dtype: object

Row 2, column 0: Ahmed

Rows 1-3 (4 excluded), first 2 columns:
     name  age
1    Sara   25
2   Ahmed   19
3  Fatima   28

Rows 0, 2, 5:
     name  age       city  gpa major
0     Ali   22     Lahore  3.5    CS
2   Ahmed   19  Islamabad  2.9    CS
5  Ayesha   23   Peshawar  3.7    EE


### When to use which

- Use **`.loc`** when working with column names (this is the common case).
- Use **`.iloc`** when you specifically want "the first 5 rows" or "the last column" by position.
- For most everyday work, `.loc` is the one you'll reach for.

### A custom index makes the difference more obvious

In [7]:
# Set 'name' as the index
df_named = df.set_index('name')
print(df_named)

        age       city  gpa major
name                             
Ali      22     Lahore  3.5    CS
Sara     25    Karachi  3.8    EE
Ahmed    19  Islamabad  2.9    CS
Fatima   28     Lahore  3.6    ME
Hassan   21    Karachi  3.1    CS
Ayesha   23   Peshawar  3.7    EE
Bilal    20     Lahore  2.8    ME
Maryam   24    Karachi  3.9    CS


In [8]:
# Now .loc uses names, .iloc uses positions
print("Ali's row by label:")
print(df_named.loc['Ali'])

print("\nRow at position 0:")
print(df_named.iloc[0])

# Mixing them up causes errors
try:
    df_named.loc[0]   # 0 isn't a label here
except KeyError as e:
    print(f"\nError trying df_named.loc[0]: {e}")

Ali's row by label:
age          22
city     Lahore
gpa         3.5
major        CS
Name: Ali, dtype: object

Row at position 0:
age          22
city     Lahore
gpa         3.5
major        CS
Name: Ali, dtype: object

Error trying df_named.loc[0]: 0


## Boolean Masking — Filtering by Condition

This is how you say "give me all rows where X is true." Just like NumPy, but applied to DataFrames.

In [9]:
# Get students with GPA > 3.5
print("Students with GPA > 3.5:")
print(df[df['gpa'] > 3.5])

Students with GPA > 3.5:
     name  age      city  gpa major
1    Sara   25   Karachi  3.8    EE
3  Fatima   28    Lahore  3.6    ME
5  Ayesha   23  Peshawar  3.7    EE
7  Maryam   24   Karachi  3.9    CS


In [10]:
# Step by step — see how it works
mask = df['gpa'] > 3.5
print("The mask (a Series of booleans):")
print(mask)

print("\nApply the mask to filter rows:")
print(df[mask])

The mask (a Series of booleans):
0    False
1     True
2    False
3     True
4    False
5     True
6    False
7     True
Name: gpa, dtype: bool

Apply the mask to filter rows:
     name  age      city  gpa major
1    Sara   25   Karachi  3.8    EE
3  Fatima   28    Lahore  3.6    ME
5  Ayesha   23  Peshawar  3.7    EE
7  Maryam   24   Karachi  3.9    CS


### Combining conditions

You **MUST** use `&` / `|` / `~`, NOT `and` / `or` / `not`. Same as NumPy. Always wrap each condition in parentheses.

In [35]:
# AND: students with GPA > 3 AND age < 25
print("GPA > 3 AND age < 25:")
print(df[(df['gpa'] > 3) & (df['age'] < 25)])

GPA > 3 AND age < 25:
     name  age      city  gpa major
0     Ali   22    Lahore  3.5    CS
4  Hassan   21   Karachi  3.1    CS
5  Ayesha   23  Peshawar  3.7    EE
7  Maryam   24   Karachi  3.9    CS


In [12]:
# OR: students in Lahore OR with GPA > 3.7
print("In Lahore OR GPA > 3.7:")
print(df[(df['city'] == 'Lahore') | (df['gpa'] > 3.7)])

In Lahore OR GPA > 3.7:
     name  age     city  gpa major
0     Ali   22   Lahore  3.5    CS
1    Sara   25  Karachi  3.8    EE
3  Fatima   28   Lahore  3.6    ME
6   Bilal   20   Lahore  2.8    ME
7  Maryam   24  Karachi  3.9    CS


In [13]:
# NOT: students NOT studying CS
print("Not in CS:")
print(df[~(df['major'] == 'CS')])

# Equivalent — using != directly
print("\nSame with !=:")
print(df[df['major'] != 'CS'])

Not in CS:
     name  age      city  gpa major
1    Sara   25   Karachi  3.8    EE
3  Fatima   28    Lahore  3.6    ME
5  Ayesha   23  Peshawar  3.7    EE
6   Bilal   20    Lahore  2.8    ME

Same with !=:
     name  age      city  gpa major
1    Sara   25   Karachi  3.8    EE
3  Fatima   28    Lahore  3.6    ME
5  Ayesha   23  Peshawar  3.7    EE
6   Bilal   20    Lahore  2.8    ME


### `isin()` — match against a list of values

In [14]:
# Students in either Lahore or Karachi
print("Students in Lahore or Karachi:")
print(df[df['city'].isin(['Lahore', 'Karachi'])])

Students in Lahore or Karachi:
     name  age     city  gpa major
0     Ali   22   Lahore  3.5    CS
1    Sara   25  Karachi  3.8    EE
3  Fatima   28   Lahore  3.6    ME
4  Hassan   21  Karachi  3.1    CS
6   Bilal   20   Lahore  2.8    ME
7  Maryam   24  Karachi  3.9    CS


In [15]:
# The negated version
print("Students NOT in Lahore or Karachi:")
print(df[~df['city'].isin(['Lahore', 'Karachi'])])

Students NOT in Lahore or Karachi:
     name  age       city  gpa major
2   Ahmed   19  Islamabad  2.9    CS
5  Ayesha   23   Peshawar  3.7    EE


### `between()` — value falls in a range

In [16]:
# Age between 22 and 25 (inclusive on both sides by default)
print("Students aged 22-25:")
print(df[df['age'].between(22, 25)])

# Exclusive on both sides
print("\nStudents aged 22-25 (exclusive):")
print(df[df['age'].between(22, 25, inclusive='neither')])

Students aged 22-25:
     name  age      city  gpa major
0     Ali   22    Lahore  3.5    CS
1    Sara   25   Karachi  3.8    EE
5  Ayesha   23  Peshawar  3.7    EE
7  Maryam   24   Karachi  3.9    CS

Students aged 22-25 (exclusive):
     name  age      city  gpa major
5  Ayesha   23  Peshawar  3.7    EE
7  Maryam   24   Karachi  3.9    CS


### Combining filtering with column selection

In [17]:
# Get only the name and GPA of students with GPA > 3.5
print("Top students' name and GPA:")
print(df.loc[df['gpa'] > 3.5, ['name', 'gpa']])

# The .loc syntax: .loc[rows, columns]
# Here rows = boolean mask, columns = list of column names

Top students' name and GPA:
     name  gpa
1    Sara  3.8
3  Fatima  3.6
5  Ayesha  3.7
7  Maryam  3.9


## `.query()` — SQL-like Filtering

For complex filters, the syntax can get ugly with all those parentheses. `.query()` lets you write the condition as a string.

In [18]:
# Traditional boolean masking
result1 = df[(df['gpa'] > 3) & (df['age'] < 25) & (df['city'] == 'Lahore')]

# Using .query() — much cleaner
result2 = df.query("gpa > 3 and age < 25 and city == 'Lahore'")

print("Both produce the same result:")
print(result2)
print(f"\nEqual? {result1.equals(result2)}")

Both produce the same result:
  name  age    city  gpa major
0  Ali   22  Lahore  3.5    CS

Equal? True


In [19]:
# Query supports all standard operators and is much more readable
print(df.query("major == 'CS' and gpa >= 3.5"))

     name  age     city  gpa major
0     Ali   22   Lahore  3.5    CS
7  Maryam   24  Karachi  3.9    CS


In [20]:
# Reference Python variables with @
min_gpa = 3.5
print(df.query("gpa >= @min_gpa"))

     name  age      city  gpa major
0     Ali   22    Lahore  3.5    CS
1    Sara   25   Karachi  3.8    EE
3  Fatima   28    Lahore  3.6    ME
5  Ayesha   23  Peshawar  3.7    EE
7  Maryam   24   Karachi  3.9    CS


**When to use `.query()`:**
- Complex multi-condition filters (more readable)
- When pasting from SQL or copy-pasting from team members
- Slightly faster for very large DataFrames

**When to stick with boolean masks:**
- Simple single-condition filters
- When you need maximum flexibility
- When working with non-standard column names (spaces, special chars)

## Real-World Selection Patterns

Things you'll do all the time in actual work.

### Pattern 1: Get rows where a specific condition holds, looking at just a few columns

In [21]:
# Show only relevant info for students who need academic help
print("Students needing help (GPA < 3.0):")
print(df.loc[df['gpa'] < 3.0, ['name', 'city', 'gpa']])

Students needing help (GPA < 3.0):
    name       city  gpa
2  Ahmed  Islamabad  2.9
6  Bilal     Lahore  2.8


### Pattern 2: Top N rows by some metric

In [22]:
# Top 3 by GPA
print("Top 3 students:")
print(df.nlargest(3, 'gpa'))

# Bottom 3 by GPA
print("\nBottom 3 students:")
print(df.nsmallest(3, 'gpa'))

Top 3 students:
     name  age      city  gpa major
7  Maryam   24   Karachi  3.9    CS
1    Sara   25   Karachi  3.8    EE
5  Ayesha   23  Peshawar  3.7    EE

Bottom 3 students:
     name  age       city  gpa major
6   Bilal   20     Lahore  2.8    ME
2   Ahmed   19  Islamabad  2.9    CS
4  Hassan   21    Karachi  3.1    CS


### Pattern 3: Random sampling

In [23]:
# Random sample of 3 rows
print("Random 3 students:")
print(df.sample(n=3, random_state=42))

# A fraction (50% of rows)
print("\nRandom 50% sample:")
print(df.sample(frac=0.5, random_state=0))

Random 3 students:
     name  age      city  gpa major
1    Sara   25   Karachi  3.8    EE
5  Ayesha   23  Peshawar  3.7    EE
0     Ali   22    Lahore  3.5    CS

Random 50% sample:
     name  age       city  gpa major
6   Bilal   20     Lahore  2.8    ME
2   Ahmed   19  Islamabad  2.9    CS
1    Sara   25    Karachi  3.8    EE
7  Maryam   24    Karachi  3.9    CS


### Pattern 4: Get unique combinations

In [24]:
# Unique (city, major) combinations
print("Unique city-major combos:")
print(df[['city', 'major']].drop_duplicates())

Unique city-major combos:
        city major
0     Lahore    CS
1    Karachi    EE
2  Islamabad    CS
3     Lahore    ME
4    Karachi    CS
5   Peshawar    EE


## The Notorious `SettingWithCopyWarning`

You WILL see this warning. It confuses every pandas user. Let's understand it once.

### The problem

In [36]:
# Filter to get Lahore students
lahore = df[df['city'] == 'Lahore']
print(lahore)

# Now try to modify the filtered subset
import warnings
warnings.simplefilter('default')

# This often triggers SettingWithCopyWarning
lahore['city'] = 'LAHORE'   #  pandas isn't sure if you meant to modify lahore or df

print("\nAfter modification:")
print(lahore)

     name  age    city  gpa major
0     Ali   22  Lahore  3.5    CS
3  Fatima   28  Lahore  3.6    ME
6   Bilal   20  Lahore  2.8    ME

After modification:
     name  age    city  gpa major
0     Ali   22  LAHORE  3.5    CS
3  Fatima   28  LAHORE  3.6    ME
6   Bilal   20  LAHORE  2.8    ME


C:\Users\hp\AppData\Local\Temp\ipykernel_59632\4090209442.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lahore['city'] = 'LAHORE'   #  pandas isn't sure if you meant to modify lahore or df


### Why this happens

When you do `df[df['city'] == 'Lahore']`, pandas isn't sure whether to return:
- A **view** (so changes affect `df`)
- A **copy** (so changes only affect `lahore`)

This ambiguity is the source of the warning. Sometimes it returns a view, sometimes a copy — and you can't easily predict which.

### The fix: be explicit

In [26]:
# Option 1: Use .loc directly to modify the original
df_copy = df.copy()
df_copy.loc[df_copy['city'] == 'Lahore', 'city'] = 'LAHORE'
print("After .loc modification:")
print(df_copy)

After .loc modification:
     name  age       city  gpa major
0     Ali   22     LAHORE  3.5    CS
1    Sara   25    Karachi  3.8    EE
2   Ahmed   19  Islamabad  2.9    CS
3  Fatima   28     LAHORE  3.6    ME
4  Hassan   21    Karachi  3.1    CS
5  Ayesha   23   Peshawar  3.7    EE
6   Bilal   20     LAHORE  2.8    ME
7  Maryam   24    Karachi  3.9    CS


In [27]:
# Option 2: Explicit copy when you want a separate object
lahore = df[df['city'] == 'Lahore'].copy()  # ← explicit copy
lahore['city'] = 'LAHORE'  # no warning, no ambiguity

print("Modified copy:")
print(lahore)
print("\nOriginal unchanged:")
print(df[df['city'] == 'Lahore'])

Modified copy:
     name  age    city  gpa major
0     Ali   22  LAHORE  3.5    CS
3  Fatima   28  LAHORE  3.6    ME
6   Bilal   20  LAHORE  2.8    ME

Original unchanged:
     name  age    city  gpa major
0     Ali   22  Lahore  3.5    CS
3  Fatima   28  Lahore  3.6    ME
6   Bilal   20  Lahore  2.8    ME


### The rule

- **Want to modify the original?** Use `df.loc[mask, col] = value`
- **Want a separate object?** Use `subset = df[mask].copy()`

Stick to those two patterns and you'll never see the warning again.

## Some Useful One-Liners

In [28]:
# First N rows by a sort
print("Youngest 3:")
print(df.sort_values('age').head(3))

# Last N rows
print("\nLast 3 in alphabetical order by name:")
print(df.sort_values('name').tail(3))

Youngest 3:
     name  age       city  gpa major
2   Ahmed   19  Islamabad  2.9    CS
6   Bilal   20     Lahore  2.8    ME
4  Hassan   21    Karachi  3.1    CS

Last 3 in alphabetical order by name:
     name  age     city  gpa major
4  Hassan   21  Karachi  3.1    CS
7  Maryam   24  Karachi  3.9    CS
1    Sara   25  Karachi  3.8    EE


## Practice Exercises

Use this slightly larger dataset for the exercises:

In [29]:
# Setup for exercises
employees = pd.DataFrame({
    'name':       ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Henry', 'Ivy', 'Jack'],
    'department': ['Eng', 'Sales', 'Eng', 'HR', 'Eng', 'Sales', 'HR', 'Eng', 'Sales', 'HR'],
    'salary':     [85000, 65000, 95000, 55000, 75000, 70000, 60000, 110000, 80000, 58000],
    'years':      [3, 5, 8, 2, 4, 6, 3, 12, 7, 1],
    'city':       ['NYC', 'LA', 'NYC', 'Chicago', 'NYC', 'LA', 'Chicago', 'NYC', 'LA', 'Chicago']
})
print(employees)

      name department  salary  years     city
0    Alice        Eng   85000      3      NYC
1      Bob      Sales   65000      5       LA
2  Charlie        Eng   95000      8      NYC
3    Diana         HR   55000      2  Chicago
4      Eve        Eng   75000      4      NYC
5    Frank      Sales   70000      6       LA
6    Grace         HR   60000      3  Chicago
7    Henry        Eng  110000     12      NYC
8      Ivy      Sales   80000      7       LA
9     Jack         HR   58000      1  Chicago


### Exercise 1
Use `.loc` to get the row(s) for employees named Alice and Diana, showing only their name, department, and salary.

### Exercise 2
Find all employees who:
- Work in Engineering AND
- Have a salary above $80,000

Show their name and salary only.

### Exercise 3
Use `.query()` to find employees who work in NYC or LA with more than 3 years of experience.

### Exercise 4
Using `isin()`, find employees in NYC or Chicago who earn more than $70,000.

### Exercise 5 (challenge)
- Top 3 highest-paid employees (use `nlargest`)
- For these 3 employees only, increase their salary by 10% (without triggering SettingWithCopyWarning)
- Print the resulting DataFrame

In [30]:
# Exercise 1
result = employees.loc[employees['name'].isin(['Alice', 'Diana']), ['name', 'department', 'salary']]
print(result)

    name department  salary
0  Alice        Eng   85000
3  Diana         HR   55000


In [31]:
# Exercise 2
result = employees.loc[
    (employees['department'] == 'Eng') & (employees['salary'] > 80000),
    ['name', 'salary']
]
print(result)

      name  salary
0    Alice   85000
2  Charlie   95000
7    Henry  110000


In [32]:
# Exercise 3
result = employees.query("(city == 'NYC' or city == 'LA') and years > 3")
print(result)

# Equivalent using isin:
result2 = employees.query("city in ['NYC', 'LA'] and years > 3")
print("\nSame with `in`:")
print(result2)

      name department  salary  years city
1      Bob      Sales   65000      5   LA
2  Charlie        Eng   95000      8  NYC
4      Eve        Eng   75000      4  NYC
5    Frank      Sales   70000      6   LA
7    Henry        Eng  110000     12  NYC
8      Ivy      Sales   80000      7   LA

Same with `in`:
      name department  salary  years city
1      Bob      Sales   65000      5   LA
2  Charlie        Eng   95000      8  NYC
4      Eve        Eng   75000      4  NYC
5    Frank      Sales   70000      6   LA
7    Henry        Eng  110000     12  NYC
8      Ivy      Sales   80000      7   LA


In [33]:
# Exercise 4
result = employees[
    employees['city'].isin(['NYC', 'Chicago']) & (employees['salary'] > 70000)
]
print(result)

      name department  salary  years city
0    Alice        Eng   85000      3  NYC
2  Charlie        Eng   95000      8  NYC
4      Eve        Eng   75000      4  NYC
7    Henry        Eng  110000     12  NYC


In [34]:
# Exercise 5
# Identify top 3 by salary
top3 = employees.nlargest(3, 'salary')
print("Top 3:")
print(top3)

# Get their names to safely modify the original
top3_names = top3['name'].tolist()

# Make a copy, convert salary to float so we can apply a 10% raise without dtype issues
emp_updated = employees.copy()
emp_updated['salary'] = emp_updated['salary'].astype(float)
emp_updated.loc[emp_updated['name'].isin(top3_names), 'salary'] *= 1.10

print("\nAfter 10% raise for top 3:")
print(emp_updated)

Top 3:
      name department  salary  years city
7    Henry        Eng  110000     12  NYC
2  Charlie        Eng   95000      8  NYC
0    Alice        Eng   85000      3  NYC

After 10% raise for top 3:
      name department    salary  years     city
0    Alice        Eng   93500.0      3      NYC
1      Bob      Sales   65000.0      5       LA
2  Charlie        Eng  104500.0      8      NYC
3    Diana         HR   55000.0      2  Chicago
4      Eve        Eng   75000.0      4      NYC
5    Frank      Sales   70000.0      6       LA
6    Grace         HR   60000.0      3  Chicago
7    Henry        Eng  121000.0     12      NYC
8      Ivy      Sales   80000.0      7       LA
9     Jack         HR   58000.0      1  Chicago
